In [1]:
import numpy as np
import random
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim

## Environment

In [2]:
class TempEnv:
    def __init__(self, target=25):
        self.T_env = 22
        self.alpha = 0.6
        self.beta = 0.015
        self.actions = [-1, -0.5, 0, 0.5, 1]
        self.target = target

    def reset(self):
        self.T = np.random.uniform(10, 40)
        return np.array([self.T], dtype=np.float32)

    def step(self, action_idx):
        power = self.actions[action_idx]
        noise = np.random.normal(0, 0.2)
        self.T = self.T + self.alpha *power - self.beta * (self.T - self.T_env) + noise
        reward = -abs(self.T - self.target) - 0.1*abs(power)

        done = False
        return np.array([self.T], dtype=np.float32), reward, done

## Replay Buffer

In [3]:
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.float32),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)

## Q-Network

In [4]:
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.net(x)

## Hyperparameters

In [5]:
env = TempEnv()

state_dim = 1
action_dim = 5

policy_net = DQN(state_dim, action_dim)
target_net = DQN(state_dim, action_dim)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
criterion = nn.MSELoss()

buffer = ReplayBuffer(capacity=10000)

gamma = 0.95
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995

batch_size = 64
episodes = 500
steps_per_episode = 200
target_update_freq = 10

episode_rewards = []

## Training Loop

In [ ]:
for episode in range(episodes):
    state = env.reset()
    total_reward = 0

    for step in range(steps_per_episode):
        if random.random() < epsilon:
            action = random.randint(0, action_dim - 1)
        else:
            with torch.no_grad():
                state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
                action = torch.argmax(policy_net(state_tensor)).item()

        next_state, reward, done = env.step(action)

        buffer.push(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward

        #Train only if enough samples are in replay buffer
        if len(buffer) >= batch_size:
            states, actions, rewards, next_states, dones = buffer.sample(batch_size)
            states = torch.tensor(states, dtype=torch.float32)
            actions = torch.tensor(actions, dtype=torch.int64).unsqueeze(1)
            rewards = torch.tensor(rewards, dtype=torch.float32).unsqueeze(1)
            next_states = torch.tensor(next_states, dtype=torch.float32)
            dones = torch.tensor(dones, dtype=torch.float32).unsqueeze(1)

            #current Q-values
            current_q = policy_net(states).gather(1, actions)

            #Target Q-values
            with torch.no_grad():
                max_next_q = target_net(next_states).max(1, keepdim=True)[0]
                target_q = rewards + gamma * max_next_q * (1 - dones)

            loss = criterion(current_q, target_q)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if done:
            break

    episode_rewards.append(total_reward)

    #Update target network
    if episode % target_update_freq == 0:
        target_net.load_state_dict(policy_net.state_dict())

    #decay epsilon
    epsilon = max(epsilon_min, epsilon*epsilon_decay)

    if episode % 20 == 0:
        print(f"Episode {episode}, Total Reward: {total_reward:.2f}, Epsilon: {epsilon:.3f}")
        

Episode 0, Total Reward: -724.69, Epsilon: 0.995
Episode 20, Total Reward: -616.90, Epsilon: 0.900
Episode 40, Total Reward: -175.90, Epsilon: 0.814
